# XClinVision: Model Comparison
**Day 3: Model Training & Comparison**

This notebook compares the performance of different model architectures:
- EfficientNet-B2
- ResNet-50
- Swin Transformer
- BiomedCLIP

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

## 2. Model Metadata Comparison

In [ ]:
# Load model configs
import yaml

MODELS = ["efficientnet_b2", "resnet50", "swin_t", "biomedclip"]

model_data = []

for model_name in MODELS:
    config_path = Path(f"../configs/{model_name}.yaml")
    if config_path.exists():
        with open(config_path) as f:
            config = yaml.safe_load(f)
        
        model_data.append({
            "Model": model_name,
            "Type": config.get("type", "unknown"),
            "Input Size": config.get("input", {}).get("size", [0, 0]),
            "Batch Size": config.get("training", {}).get("batch_size", 0),
            "Expected Recall": config.get("comparison", {}).get("expected_performance", {}).get("recall_pneumonia", 0),
            "Expected ECE": config.get("comparison", {}).get("expected_performance", {}).get("ece", 0),
            "Category": config.get("comparison", {}).get("category", "unknown"),
        })

df_models = pd.DataFrame(model_data)
print("Model Configuration Summary:")
print(df_models.to_string(index=False))

## 3. Expected Performance Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Expected Recall
sns.barplot(data=df_models, x="Model", y="Expected Recall", hue="Category", ax=axes[0])
axes[0].set_title("Expected Pneumonia Recall")
axes[0].set_ylim(0.8, 1.0)
for i, row in df_models.iterrows():
    axes[0].text(i, row["Expected Recall"] + 0.01, f"{row['Expected Recall']:.2f}", ha="center")

# Expected ECE
sns.barplot(data=df_models, x="Model", y="Expected ECE", hue="Category", ax=axes[1])
axes[1].set_title("Expected Calibration Error (ECE)")
axes[1].set_ylim(0, 0.2)
for i, row in df_models.iterrows():
    axes[1].text(i, row["Expected ECE"] + 0.005, f"{row['Expected ECE']:.2f}", ha="center")

plt.tight_layout()
plt.show()

## 4. Model Strengths & Weaknesses

In [ ]:
for model_name in MODELS:
    config_path = Path(f"../configs/{model_name}.yaml")
    if config_path.exists():
        with open(config_path) as f:
            config = yaml.safe_load(f)
        
        print(f"\n{'='*60}")
        print(f"📊 {model_name.upper()}")
        print(f"{'='*60}")
        
        strengths = config.get("comparison", {}).get("strengths", [])
        weaknesses = config.get("comparison", {}).get("weaknesses", [])
        
        print("\n✅ Strengths:")
        for s in strengths:
            print(f"  • {s}")
        
        print("\n⚠️ Weaknesses:")
        for w in weaknesses:
            print(f"  • {w}")

## 5. Architecture Parameter Count (Estimated)

In [ ]:
# Parameter counts (approximate)
param_counts = {
    "efficientnet_b2": "9.2M",
    "resnet50": "25.6M",
    "swin_t": "28.3M",
    "biomedclip": "86M",
}

df_models["Parameters"] = df_models["Model"].map(param_counts)

# Parse numeric values for plotting
def parse_params(p):
    if "M" in p:
        return float(p.replace("M", ""))
    return 0

df_models["Params_M"] = df_models["Parameters"].apply(parse_params)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(df_models["Params_M"], df_models["Expected Recall"], 
                    s=df_models["Expected ECE"]*5000, 
                    c=range(len(df_models)), cmap="viridis", alpha=0.7)

for i, row in df_models.iterrows():
    ax.annotate(row["Model"], (row["Params_M"], row["Expected Recall"]),
                xytext=(5, 5), textcoords="offset points")

ax.set_xlabel("Parameters (Millions)")
ax.set_ylabel("Expected Recall")
ax.set_title("Model Comparison: Recall vs Parameters (bubble size = ECE)")
plt.tight_layout()
plt.show()